### Setup

In [1]:
import numpy as np
import pandas as pd
import yfinance as yf
from scipy.stats import norm

value = 1000000
alpha = 0.97
beta = 0.02
trials = 125000

def empirical_var(losses, level):
    ordered = np.sort(losses)
    return ordered[np.ceil(len(ordered)*level).astype(int) - 1]

def normal_var(m, s):
    return value*(1 - np.exp(m + s*norm.ppf(1 - alpha)))

prices = yf.download("AAPL", start="2016-11-01", end="2018-11-01", auto_adjust=False)["Close"].squeeze()
returns = np.log(prices).diff().dropna()
losses = -value*(np.exp(returns) - 1)
print("prices", len(prices))
print("returns", len(returns))

[*********************100%***********************]  1 of 1 completed

prices 504
returns 503


### a.

In [2]:
from scipy.stats import chi2

n = len(returns)
mean = returns.mean()
sd = returns.std(ddof=1)

var_emp = empirical_var(losses.values, alpha)
var_theory = normal_var(mean, sd)

low_sd = np.sqrt((n - 1)/chi2.ppf(1 - beta/2, n - 1))*sd
high_sd = np.sqrt((n - 1)/chi2.ppf(beta/2, n - 1))*sd
var_low = normal_var(mean, low_sd)
var_high = normal_var(mean, high_sd)

print("days", n)
print("sample mean", round(mean, 6), "sample sd", round(sd, 6))
print("empirical VaR", round(var_emp, 2))
print("theoretical VaR", round(var_theory, 2))
print("sigma band", round(low_sd, 6), round(high_sd, 6))
print("confidence interval", round(var_low, 2), round(var_high, 2))

days 503
sample mean 0.001341 sample sd 0.012961
empirical VaR 24984.39
theoretical VaR 22773.55
sigma band 0.012073 0.013984
confidence interval 21138.53 24651.99


### b.

In [3]:
generator = np.random.default_rng(42)

draws = generator.chisquare(n - 1, trials)
sd_draws = np.sqrt((n - 1)*sd**2/draws)
mean_draws = generator.normal(mean, sd_draws/np.sqrt(n))
sample = normal_var(mean_draws, sd_draws)

print("average VaR", round(sample.mean(), 2))
print("confidence interval", round(np.quantile(sample, beta/2), 2), round(np.quantile(sample, 1 - beta/2), 2))

average VaR 22809.63
confidence interval 20702.35 25101.26
